# Financial SMS Model 1 — Logistic Regression vs Multinomial Naive Bayes

A fair comparison of the two algorithms used for Model 1.

**Algorithms**
1. Logistic Regression
2. Multinomial Naive Bayes

Both use the same TF-IDF settings, the same cleaned dataset, and the same stratified 80/20 train-test split.

The notebook reports Accuracy, Precision, Transaction Recall, F1, confusion matrices, errors, unseen-SMS predictions, and a final recommendation.

In [ ]:
# STEP 1 — SETUP
!pip install -q pandas numpy scikit-learn openpyxl matplotlib joblib

import os, re, time, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
print("Setup complete.")

In [ ]:
# STEP 2 — LOAD DATASET
DATA_PATH = "bank_sms_rule_based_model.xlsx"

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Upload bank_sms_rule_based_model.xlsx")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No file uploaded.")
    DATA_PATH = list(uploaded.keys())[0]

xls = pd.ExcelFile(DATA_PATH)
print("Available sheets:", xls.sheet_names)

df = pd.read_excel(DATA_PATH, sheet_name="Model Predictions")
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

In [ ]:
# STEP 3 — CLEAN DATA AND CREATE LABELS
if "SMS" not in df.columns or "Type" not in df.columns:
    raise ValueError("Dataset must contain SMS and Type columns.")

df["SMS"] = df["SMS"].fillna("").astype(str).str.strip()
df["Type"] = df["Type"].fillna("").astype(str).str.strip()
df = df[df["SMS"].str.len() > 0].copy()

TRANSACTION_TYPES = {
    "Debit", "Credit", "Transfer (Debit)", "Transfer (Credit)",
    "Deposit", "Withdrawal"
}

def map_ground_truth(value):
    value = str(value).strip()
    if value in TRANSACTION_TYPES:
        return "Transaction"
    if value.startswith("Non-Transaction:") or value == "E-Ticket":
        return "Non-Transaction"
    return "AMBIGUOUS"

def normalize_sms(text):
    text = unicodedata.normalize("NFKC", str(text))
    return re.sub(r"\s+", " ", text).strip()

df["Ground_Truth_Flag_raw"] = df["Type"].apply(map_ground_truth)

print("Raw Model 1 labels:")
print(df["Ground_Truth_Flag_raw"].value_counts(dropna=False))

model_df = df[
    df["Ground_Truth_Flag_raw"].isin(["Transaction", "Non-Transaction"])
][["SMS", "Ground_Truth_Flag_raw"]].rename(
    columns={"Ground_Truth_Flag_raw": "label"}
).copy()

model_df["normalized_sms"] = model_df["SMS"].apply(normalize_sms)

conflicts = model_df.groupby("normalized_sms")["label"].nunique()
conflicts = conflicts[conflicts > 1]

print("Conflicting SMS:", len(conflicts))

model_df = model_df[
    ~model_df["normalized_sms"].isin(conflicts.index)
].drop_duplicates("normalized_sms", keep="first").reset_index(drop=True)

model_df["text"] = model_df["SMS"].apply(normalize_sms)
model_df = model_df[["SMS", "text", "label"]]

print("\nFinal dataset:", model_df.shape)
print(model_df["label"].value_counts())

In [ ]:
# STEP 4 — SAME TRAIN/TEST SPLIT FOR BOTH ALGORITHMS
X = model_df["text"]
y = model_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training:", len(X_train))
print("Testing :", len(X_test))
print("\nTest distribution:")
print(y_test.value_counts())

In [ ]:
# STEP 5 — CREATE BOTH MODELS WITH IDENTICAL TF-IDF
def make_tfidf():
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        max_features=20000
    )

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),
    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", MultinomialNB())
    ])
}

print("Both models created.")

In [ ]:
# STEP 6 — TRAIN BOTH MODELS
trained_models = {}
predictions = {}
training_times = {}

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start

    trained_models[name] = model
    predictions[name] = model.predict(X_test)
    training_times[name] = elapsed

    print(f"{name}: trained in {elapsed:.4f} sec")

In [ ]:
# STEP 7 — MAIN COMPARISON TABLE
rows = []

for name, pred in predictions.items():
    rows.append({
        "Algorithm": name,
        "Accuracy (%)": accuracy_score(y_test, pred) * 100,
        "Precision (%)": precision_score(
            y_test, pred, pos_label="Transaction", zero_division=0
        ) * 100,
        "Transaction Recall (%)": recall_score(
            y_test, pred, pos_label="Transaction", zero_division=0
        ) * 100,
        "F1 Score (%)": f1_score(
            y_test, pred, pos_label="Transaction", zero_division=0
        ) * 100,
        "Training Time (sec)": training_times[name]
    })

comparison_df = pd.DataFrame(rows)

print("=" * 90)
print("MODEL 1 — LOGISTIC REGRESSION vs MULTINOMIAL NAIVE BAYES")
print("=" * 90)

display(comparison_df.style.format({
    "Accuracy (%)": "{:.2f}",
    "Precision (%)": "{:.2f}",
    "Transaction Recall (%)": "{:.2f}",
    "F1 Score (%)": "{:.2f}",
    "Training Time (sec)": "{:.4f}"
}))

In [ ]:
# STEP 8 — CLASSIFICATION REPORTS
for name, pred in predictions.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(y_test, pred, zero_division=0))

In [ ]:
# STEP 9 — CONFUSION MATRICES
labels = ["Non-Transaction", "Transaction"]

for name, pred in predictions.items():
    cm = confusion_matrix(y_test, pred, labels=labels)
    print("\n", name)
    print(cm)

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    ).plot()
    plt.title(name)
    plt.tight_layout()
    plt.show()

In [ ]:
# STEP 10 — MISCLASSIFIED SMS FOR BOTH
for name, pred in predictions.items():
    result = pd.DataFrame({
        "SMS": X_test.values,
        "Expected": y_test.values,
        "Predicted": pred
    })

    wrong = result[result["Expected"] != result["Predicted"]].reset_index(drop=True)

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("Total test samples:", len(result))
    print("Correct:", len(result) - len(wrong))
    print("Incorrect:", len(wrong))

    if len(wrong):
        display(wrong)
    else:
        print("No misclassified SMS.")

In [ ]:
# STEP 11 — DIRECT SIDE-BY-SIDE TEST SET COMPARISON
side_by_side = pd.DataFrame({
    "SMS": X_test.values,
    "Expected": y_test.values,
    "Logistic Regression": predictions["Logistic Regression"],
    "Multinomial Naive Bayes": predictions["Multinomial Naive Bayes"]
})

side_by_side["LR Correct"] = (
    side_by_side["Expected"] == side_by_side["Logistic Regression"]
)

side_by_side["MNB Correct"] = (
    side_by_side["Expected"] == side_by_side["Multinomial Naive Bayes"]
)

display(side_by_side)

In [ ]:
# STEP 12 — UNSEEN SMS TEST
unseen_sms = [
    "Dear Sir/Madam, Your A/C 065-2001****11 has been debited by Rs. 1500.00 (ATM @10:30 28/08/2026)",
    "LKR 5000.00 credited to Ac No:01602XXXXX99 on 24/08/26 11:16:18 Reason:CEFT-APPA",
    "POS/ATM Transaction Rs 1030.00 From A/C No XXXXXXXXXX875. Balance available Rs 939.21",
    "Online Transfer Credit Rs 1500.00 To A/C No XXXXXXXXXX875.",
    "Online Transfer Debit Rs 500.00 From A/C No XXXXXXXXXX875.",
    "Congratulations! You have won exciting reward points.",
    "Get unlimited internet data today. Special promotional offer.",
    "Your loan application is eligible for a special offer.",
    "Dear customer, please do not share your OTP or account details with anyone.",
    "Your account has been credited with Rs 25000.00."
]

for i, sms in enumerate(unseen_sms, 1):
    text = normalize_sms(sms)
    print("\n" + "-" * 80)
    print("TEST", i)
    print("SMS:", sms)

    for name, model in trained_models.items():
        pred = model.predict([text])[0]

        if hasattr(model, "predict_proba"):
            confidence = np.max(model.predict_proba([text])[0]) * 100
            print(f"{name:28s} -> {pred} | Confidence: {confidence:.2f}%")
        else:
            print(f"{name:28s} -> {pred}")

In [ ]:
# STEP 13 — 100-SAMPLE STRESS TEST
# Same 100-sample stress-test structure used previously.

test_100 = [
    ("Your A/C has been debited by Rs. 1500.00 for ATM withdrawal.", "Transaction"),
    ("Your account has been credited with LKR 5000.00.", "Transaction"),
    ("POS transaction of Rs. 2500.00 was debited from your account.", "Transaction"),
    ("ATM Withdrawal Rs 5000.00 From A/C No XXXXXXXXXX875.", "Transaction"),
    ("Online Transfer Credit Rs 1500.00 To A/C No XXXXXXXXXX875.", "Transaction"),
    ("Online Transfer Debit Rs 500.00 From A/C No XXXXXXXXXX875.", "Transaction"),
    ("CEFT Transfer Debit Rs 37525.00 From A/C No XXXXXXXXXX875.", "Transaction"),
    ("ATM Cash Deposit Rs 50000.00 To A/C No XXXXXXXXXX875.", "Transaction"),
    ("LKR 10000.00 credited to your account via CEFT transfer.", "Transaction"),
    ("LKR 2500.00 debited from your account for a purchase.", "Transaction"),
    ("Your A/C has been credited by Rs. 7500.00 via cash deposit.", "Transaction"),
    ("Your account was debited by LKR 3500.00 at ATM.", "Transaction"),
    ("Your account was credited with LKR 15000.00 through transfer.", "Transaction"),
    ("Rs 1000.00 has been debited from your account.", "Transaction"),
    ("Rs 25000.00 has been credited to your account.", "Transaction"),
    ("Bank transaction successful. Amount debited Rs 450.00.", "Transaction"),
    ("Bank transaction successful. Amount credited Rs 8500.00.", "Transaction"),
    ("Your card purchase of Rs 1200.00 was successful.", "Transaction"),
    ("Your debit card was charged LKR 2300.00.", "Transaction"),
    ("Your account received LKR 6500.00.", "Transaction"),
    ("LKR 500.00 was transferred from your account.", "Transaction"),
    ("LKR 9000.00 was transferred to your account.", "Transaction"),
    ("Transfer Debit Rs 1500.00 From A/C No XXXXXXXXXX875.", "Transaction"),
    ("Transfer Credit Rs 3000.00 To A/C No XXXXXXXXXX875.", "Transaction"),
    ("Online transfer of LKR 5500.00 completed successfully.", "Transaction"),
    ("Online transfer credit of LKR 7000.00 completed.", "Transaction"),
    ("Online transfer debit of LKR 1800.00 completed.", "Transaction"),
    ("Cash deposit of Rs 20000.00 was successful.", "Transaction"),
    ("Cash withdrawal of Rs 5000.00 was successful.", "Transaction"),
    ("ATM cash withdrawal Rs 2500.00 completed.", "Transaction"),
    ("ATM deposit Rs 10000.00 credited to your account.", "Transaction"),
    ("POS purchase Rs 750.00 debited from account.", "Transaction"),
    ("POS payment of LKR 4200.00 completed successfully.", "Transaction"),
    ("Your account has been debited for LKR 999.00.", "Transaction"),
    ("Your account has been credited for LKR 12500.00.", "Transaction"),
    ("A/C 123456 has been debited by Rs 3500.00.", "Transaction"),
    ("A/C 123456 has been credited by Rs 8500.00.", "Transaction"),
    ("LKR 1250.00 debit transaction completed.", "Transaction"),
    ("LKR 6000.00 credit transaction completed.", "Transaction"),
    ("Your transaction for Rs 2750.00 was successful.", "Transaction"),
    ("Payment of LKR 1800.00 has been processed.", "Transaction"),
    ("Your card was charged Rs 950.00.", "Transaction"),
    ("Rs 12500.00 received through bank transfer.", "Transaction"),
    ("Rs 4500.00 sent through bank transfer.", "Transaction"),
    ("CEFT credit of LKR 8500.00 received.", "Transaction"),
    ("CEFT debit of LKR 2750.00 completed.", "Transaction"),
    ("LANKAPAY transfer of LKR 5000.00 was successful.", "Transaction"),
    ("LANKAPAY transfer of LKR 2500.00 to your account was successful.", "Transaction"),
    ("Bill payment of LKR 1285.00 was successful.", "Transaction"),
    ("Utility payment of Rs 3200.00 completed successfully.", "Transaction"),

    ("Congratulations! You have won exciting reward points.", "Non-Transaction"),
    ("Claim your special reward before it expires.", "Non-Transaction"),
    ("Get unlimited internet data today at a special price.", "Non-Transaction"),
    ("Enjoy our latest mobile data promotion.", "Non-Transaction"),
    ("Special internet package available for you.", "Non-Transaction"),
    ("Congratulations! You are selected for a special offer.", "Non-Transaction"),
    ("Win exciting prizes today. Click here to participate.", "Non-Transaction"),
    ("You have been selected for an exclusive promotion.", "Non-Transaction"),
    ("Get 50% discount on your next purchase.", "Non-Transaction"),
    ("Limited time promotional offer available now.", "Non-Transaction"),
    ("Your reward points are waiting to be redeemed.", "Non-Transaction"),
    ("Redeem your loyalty points before they expire.", "Non-Transaction"),
    ("Special bonus offer available for selected customers.", "Non-Transaction"),
    ("Enjoy special discounts and promotional benefits.", "Non-Transaction"),
    ("Subscribe now and receive extra internet data.", "Non-Transaction"),
    ("Activate your special data package today.", "Non-Transaction"),
    ("Your promotional package is available now.", "Non-Transaction"),
    ("Call us to learn about our latest offer.", "Non-Transaction"),
    ("Exclusive customer offer available this week.", "Non-Transaction"),
    ("Congratulations on being selected for a prize.", "Non-Transaction"),
    ("Dear customer, please do not share your OTP with anyone.", "Non-Transaction"),
    ("Never share your account password or OTP with anyone.", "Non-Transaction"),
    ("For your security, do not disclose your OTP.", "Non-Transaction"),
    ("Protect your account. Never share your PIN.", "Non-Transaction"),
    ("Bank security notice: never share your account details.", "Non-Transaction"),
    ("Please be aware of online banking scams.", "Non-Transaction"),
    ("Do not provide your banking credentials to anyone.", "Non-Transaction"),
    ("Your bank will never ask for your OTP.", "Non-Transaction"),
    ("Security reminder: keep your banking information private.", "Non-Transaction"),
    ("Please contact the bank if you suspect a scam.", "Non-Transaction"),
    ("Your loan application is eligible for a special offer.", "Non-Transaction"),
    ("Get a personal loan with our special promotion.", "Non-Transaction"),
    ("Special loan interest rate available now.", "Non-Transaction"),
    ("Apply today for our exclusive loan offer.", "Non-Transaction"),
    ("You are pre-approved for a special loan promotion.", "Non-Transaction"),
    ("Get financial assistance with our new loan package.", "Non-Transaction"),
    ("Special credit card offer available for you.", "Non-Transaction"),
    ("Apply for our new credit card promotion.", "Non-Transaction"),
    ("Enjoy exclusive banking benefits with our new offer.", "Non-Transaction"),
    ("Visit your nearest branch for more information.", "Non-Transaction"),
    ("Your account services will be temporarily unavailable.", "Non-Transaction"),
    ("Scheduled maintenance will affect online banking services.", "Non-Transaction"),
    ("Our mobile banking service will undergo maintenance.", "Non-Transaction"),
    ("Please update your contact information with the bank.", "Non-Transaction"),
    ("Your banking application requires an update.", "Non-Transaction"),
    ("Thank you for banking with us. Please review our security tips.", "Non-Transaction"),
    ("Important customer notice regarding banking security.", "Non-Transaction"),
    ("Please read our latest customer service announcement.", "Non-Transaction"),
    ("E-ticket confirmation for your upcoming journey.", "Non-Transaction"),
    ("Your bus ticket has been successfully booked.", "Non-Transaction")
]

assert len(test_100) == 100

stress_rows = []

for name, model in trained_models.items():
    pred = model.predict([normalize_sms(x[0]) for x in test_100])
    expected = [x[1] for x in test_100]

    stress_rows.append({
        "Algorithm": name,
        "Correct": int(np.sum(pred == np.array(expected))),
        "Incorrect": int(np.sum(pred != np.array(expected))),
        "Accuracy (%)": accuracy_score(expected, pred) * 100,
        "Precision (%)": precision_score(
            expected, pred, pos_label="Transaction", zero_division=0
        ) * 100,
        "Transaction Recall (%)": recall_score(
            expected, pred, pos_label="Transaction", zero_division=0
        ) * 100,
        "F1 Score (%)": f1_score(
            expected, pred, pos_label="Transaction", zero_division=0
        ) * 100
    })

stress_comparison_df = pd.DataFrame(stress_rows)

print("=" * 90)
print("100-SAMPLE STRESS TEST COMPARISON")
print("=" * 90)

display(stress_comparison_df.style.format({
    "Accuracy (%)": "{:.2f}",
    "Precision (%)": "{:.2f}",
    "Transaction Recall (%)": "{:.2f}",
    "F1 Score (%)": "{:.2f}"
}))

In [ ]:
# STEP 14 — FINAL RECOMMENDATION
# For this project, prioritize:
# 1. Transaction Recall
# 2. F1 Score
# 3. Accuracy

decision_df = comparison_df.sort_values(
    by=["Transaction Recall (%)", "F1 Score (%)", "Accuracy (%)"],
    ascending=False
).reset_index(drop=True)

recommended = decision_df.iloc[0]["Algorithm"]

print("=" * 90)
print("MODEL 1 RECOMMENDATION")
print("=" * 90)
print("Recommended algorithm:", recommended)
print()
print("Priority:")
print("1. Transaction Recall")
print("2. F1 Score")
print("3. Accuracy")

display(decision_df.style.format({
    "Accuracy (%)": "{:.2f}",
    "Precision (%)": "{:.2f}",
    "Transaction Recall (%)": "{:.2f}",
    "F1 Score (%)": "{:.2f}",
    "Training Time (sec)": "{:.4f}"
}))

In [ ]:
# STEP 15 — SAVE COMPARISON RESULTS
comparison_df.to_csv(
    "model1_logistic_vs_mnb_comparison.csv",
    index=False
)

stress_comparison_df.to_csv(
    "model1_logistic_vs_mnb_100_stress_comparison.csv",
    index=False
)

side_by_side.to_csv(
    "model1_logistic_vs_mnb_test_predictions.csv",
    index=False
)

print("Saved:")
print("- model1_logistic_vs_mnb_comparison.csv")
print("- model1_logistic_vs_mnb_100_stress_comparison.csv")
print("- model1_logistic_vs_mnb_test_predictions.csv")